In [1]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py

--2025-12-07 15:48:14--  https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3668 (3.6K) [text/plain]
Saving to: ‘sylbreak.py’

sylbreak.py         100%[===================>]   3.58K  --.-KB/s    in 0s      

2025-12-07 15:48:15 (48.3 MB/s) - ‘sylbreak.py’ saved [3668/3668]



In [2]:
!pip install 'datasets[audio]==2.14.4' 'fsspec==2023.9.2'

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
  Attempting uninstall: datasets
    Fou

In [3]:
!pip install -q transformers datasets librosa evaluate jiwer gradio bitsandbytes accelerate
!pip install -q git+https://github.com/huggingface/peft.git@main

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/

In [4]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
model_name_or_path = "openai/whisper-large-v2"
#model_name_or_path = "openai/whisper-tiny"
task = "transcribe"

In [6]:
dataset_name = "LULab/mediTalk-mm-rdy"
language = "myanmar"
language_abbr = "my" # Short hand code for the language we want to fine-tune

In [8]:
from datasets import load_dataset, DatasetDict

speech_data = load_dataset(dataset_name, streaming=True)
speech_data

{'test': <datasets.iterable_dataset.IterableDataset at 0x7f11131026d0>,
 'train': <datasets.iterable_dataset.IterableDataset at 0x7f10260b0490>}

In [9]:
from sylbreak import break_syllables, create_break_pattern

def syllable_break(text):
  """Syllable break for burmese texts"""
  text = text
  separator = ' '
  break_pattern = create_break_pattern()

  segmented = break_syllables(text, break_pattern, separator)
  return segmented

In [10]:
def apply_syllable_break(text):
    text['prompt'] = syllable_break(text['prompt'])
    return text

dataset = speech_data.map(apply_syllable_break)
dataset

{'test': <datasets.iterable_dataset.IterableDataset at 0x7f11131335d0>,
 'train': <datasets.iterable_dataset.IterableDataset at 0x7f11131021d0>}

In [11]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_or_path)

preprocessor_config.json: 0.00B [00:00, ?B/s]

In [12]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(model_name_or_path, language=language, task=task)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [13]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(model_name_or_path, language=language, task=task)

2025-12-07 15:50:18.352136: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765122618.555614      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765122618.614037      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [14]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["prompt"], truncation=True, max_length=224).input_ids
    return batch

In [15]:
processed_dataset = dataset.map(prepare_dataset , remove_columns = list(next(iter(speech_data.values())).features)).with_format("torch")

In [16]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [17]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [18]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path, device_map="auto", load_in_8bit=True)

config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [19]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [20]:
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model

config = LoraConfig(
    r=128,                         
    
    lora_alpha=256,               
    lora_dropout=0.05,             

    target_modules=["q_proj", "v_proj"],

    bias="none",

    use_rslora=True,          

    fan_in_fan_out=False,
    modules_to_save=None,
    init_lora_weights=True,
    use_dora=False
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 62,914,560 || all params: 1,606,219,520 || trainable%: 3.9169


In [21]:
# trainable params: 7,077,888 || all params: 248,812,800 || trainable%: 2.8447


In [22]:
import evaluate

metric = evaluate.load("wer")

In [23]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-large-v2-lt1",  # change to a repo name of your choice
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=50,
    #num_train_epochs=3,
    eval_strategy="steps",
    fp16=True,
    per_device_eval_batch_size=8,
    load_best_model_at_end=True,
    #metric_for_best_model="wer",
    generation_max_length=225,
    logging_steps=25,
    save_steps=500, # default = 500
    eval_steps=500, # default = 500
    max_steps=2000, # only for testing purposes, remove this from your final run :)
    remove_unused_columns=False,  # required as the PeftModel forward doesn't have the signature of the wrapped model's forward
    push_to_hub=False,
    #hub_model_id="YeBhoneLin10/Whisper-Base-MM",
    #hub_strategy="checkpoint",
    #save_total_limit=5,
    label_names=["labels"],  # same reason as above
)

In [24]:
from transformers import Seq2SeqTrainer, TrainerCallback, TrainingArguments, TrainerState, TrainerControl
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR

# This callback helps to save only the adapter weights and remove the base model weights.
class SavePeftModelCallback(TrainerCallback):
    def on_save(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        checkpoint_folder = os.path.join(args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}")

        peft_model_path = os.path.join(checkpoint_folder, "adapter_model")
        kwargs["model"].save_pretrained(peft_model_path)

        pytorch_model_path = os.path.join(checkpoint_folder, "pytorch_model.bin")
        if os.path.exists(pytorch_model_path):
            os.remove(pytorch_model_path)
        return control


trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset['train'],
    eval_dataset= processed_dataset['test'],
    data_collator=data_collator,
    #compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[SavePeftModelCallback],
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!

/tmp/ipykernel_20/4204498947.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [26]:
trainer.train() # be8af9903463cf6f09275a33dbac9efd1653e893

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251207_155123-upa0nrj0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ./whisper-large-v2-lt1
wandb: ⭐️ View project at https://wandb.ai/yebhonelin10/huggingface
wandb: 🚀 View run at https://wandb.ai/yebhonelin10/huggingface/runs/upa0nrj0
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differenc

Step,Training Loss,Validation Loss
500,0.247300,0.287747
1000,0.147200,0.174402
1500,0.117800,0.145266
2000,0.097500,0.131253


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarni

TrainOutput(global_step=2000, training_loss=0.2578707834482193, metrics={'train_runtime': 13768.7902, 'train_samples_per_second': 0.581, 'train_steps_per_second': 0.145, 'total_flos': 1.77101438976e+19, 'train_loss': 0.2578707834482193, 'epoch': 1.0})

In [27]:
peft_model_id = "YeBhoneLin10/whisper-large-v2-peft-myanmar"
model.push_to_hub(peft_model_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/YeBhoneLin10/whisper-large-v2-peft-myanmar/commit/5b4b68c20c2eb565a55109ddaedbc4b32baf7720', commit_message='Upload model', commit_description='', oid='5b4b68c20c2eb565a55109ddaedbc4b32baf7720', pr_url=None, repo_url=RepoUrl('https://huggingface.co/YeBhoneLin10/whisper-large-v2-peft-myanmar', endpoint='https://huggingface.co', repo_type='model', repo_id='YeBhoneLin10/whisper-large-v2-peft-myanmar'), pr_revision=None, pr_num=None)